In [1]:
user_lib <- path.expand("~/R/x86_64-pc-linux-gnu-library/4.4")

if (dir.exists(user_lib)) {
  .libPaths(unique(c(user_lib, .libPaths())))
}

In [2]:
###############################################################
# 00 — Setup Check
#
# Purpose:
#   This notebook verifies that the project environment is correctly configured 
#   before any analysis is performed.
#
# The health check verifies:
#
#    - Required R packages
#    - Project directory structure
#    - Helper scripts
#    - Reference files
#    - Metadata files
#    - Sample inventory
#    - Master Sample Table
#    - Salmon quantification files
#
# Running this notebook at the start of each work session helps ensure that 
# all downstream analyses are reproducible and that the project environment 
# is functioning correctly.
#
# If any check fails, resolve the issue before continuing to the analysis notebooks.
#################################################################################

In [3]:
cat("=====================================\n")
cat("Project Health Check\n")
cat("=====================================\n\n")

cat("Checking required packages...\n\n")

Project Health Check

Checking required packages...



In [4]:
package_requirements <- data.frame(
  Package = c(
    "readxl",
    "dplyr",
    "stringr",
    "ggplot2",
    "tximport",
    "DESeq2"
  ),
  Used_For = c(
    "Reading Excel metadata",
    "Data cleaning and manipulation",
    "Text and sample-name processing",
    "Visualizations",
    "Importing Salmon quantification files",
    "Differential-expression analysis and VST"
  ),
  Stage = c(
    "Metadata",
    "Metadata",
    "Metadata",
    "Visualization",
    "RNA-seq import",
    "Differential expression"
  ),
  stringsAsFactors = FALSE
)

package_requirements$Installed <- vapply(
  package_requirements$Package,
  requireNamespace,
  quietly = TRUE,
  FUN.VALUE = logical(1)
)

package_requirements$Status <- ifelse(
  package_requirements$Installed,
  "✓ Installed",
  "✗ Missing"
)

cat("============================================================\n")
cat(" REQUIRED PACKAGES\n")
cat("============================================================\n\n")

package_display <- package_requirements[
  ,
  c("Package", "Stage", "Status")
]

print(package_display, row.names = FALSE)

 REQUIRED PACKAGES

  Package                   Stage      Status
   readxl                Metadata ✓ Installed
    dplyr                Metadata ✓ Installed
  stringr                Metadata ✓ Installed
  ggplot2           Visualization ✓ Installed
 tximport          RNA-seq import ✓ Installed
   DESeq2 Differential expression ✓ Installed


In [19]:
###############################################################################
# Missing Package Summary
###############################################################################

missing_packages <- package_requirements[
  !package_requirements$Installed,
  c("Package", "Used_For", "Stage"),
  drop = FALSE
]

packages_ok <- nrow(missing_packages) == 0

if (packages_ok) {

  cat("\n✓ All required project packages are installed.\n\n")

} else {

  cat("\n⚠ Missing required packages\n\n")

  print(missing_packages, row.names = FALSE)

  cat(
    "\nInstall the missing package(s) before running the corresponding stage(s) of the pipeline.\n\n"
  )

}


✓ All required project packages are installed.



In [20]:
###############################################################
# Load Required Packages
###############################################################

cat("PACKAGE LOADING...\n")

required_packages <- c(
  "tximport",
  "DESeq2",
  "readxl",
  "dplyr",
  "stringr",
  "ggplot2"
)

for (pkg in required_packages) {

  suppressPackageStartupMessages(
    library(pkg,
            character.only = TRUE)
  )

  cat("✓", pkg, "loaded successfully.\n")

}

PACKAGE LOADING...
✓ tximport loaded successfully.
✓ DESeq2 loaded successfully.
✓ readxl loaded successfully.
✓ dplyr loaded successfully.
✓ stringr loaded successfully.
✓ ggplot2 loaded successfully.


In [21]:
###############################################################################
# DEFINE PROJECT PATHS
###############################################################################

base_dir    <- "/standard/harrislab/capstone_2026"
project_dir <- path.expand("~/MSDS/capstone_project")

scripts_dir  <- file.path(project_dir, "scripts")
notebooks_dir <- file.path(project_dir, "notebooks")
results_dir  <- file.path(project_dir, "results")
figures_dir  <- file.path(project_dir, "figures")

metadata_results_dir <- file.path(results_dir, "metadata")
counts_dir           <- file.path(results_dir, "counts")
tpm_dir              <- file.path(results_dir, "tpm")
deseq2_dir           <- file.path(results_dir, "deseq2")
pathway_dir           <- file.path(results_dir, "pathway")

cat("PROJECT PATHS\n")
cat("Sponsor data:", base_dir, "\n")
cat("Project directory:", project_dir, "\n\n")

PROJECT PATHS
Sponsor data: /standard/harrislab/capstone_2026 
Project directory: /home/kvq4nvn/MSDS/capstone_project 



In [22]:
cat("\nChecking project directories...\n")


Checking project directories...


In [23]:
###############################################################################
# 5. DIRECTORY STRUCTURE CHECK
###############################################################################

directories <- c(
  "Sponsor Data"      = base_dir,
  "Project"           = project_dir,
  "Notebooks"         = notebooks_dir,
  "Scripts"           = scripts_dir,
  "Results"           = results_dir,
  "Metadata Results"  = metadata_results_dir,
  "Count Results"     = counts_dir,
  "TPM Results"       = tpm_dir,
  "DESeq2 Results"    = deseq2_dir,
  "Pathway Results"   = pathway_dir,
  "Figures"           = figures_dir
)

directory_status <- data.frame(
  Directory = names(directories),
  Path = unname(directories),
  Exists = vapply(directories, dir.exists, logical(1)),
  stringsAsFactors = FALSE
)

cat("DIRECTORY STATUS\n")
print(directory_status, row.names = FALSE)

directories_ok <- all(directory_status$Exists)

if (directories_ok) {
  cat("\n✓ All expected directories exist.\n\n")
} else {
  cat("\n⚠ Missing directories:\n")
  print(
    directory_status[!directory_status$Exists, ],
    row.names = FALSE
  )
  cat("\n")
}

DIRECTORY STATUS
        Directory                                                 Path Exists
     Sponsor Data                    /standard/harrislab/capstone_2026   TRUE
          Project                  /home/kvq4nvn/MSDS/capstone_project   TRUE
        Notebooks        /home/kvq4nvn/MSDS/capstone_project/notebooks   TRUE
          Scripts          /home/kvq4nvn/MSDS/capstone_project/scripts   TRUE
          Results          /home/kvq4nvn/MSDS/capstone_project/results   TRUE
 Metadata Results /home/kvq4nvn/MSDS/capstone_project/results/metadata   TRUE
    Count Results   /home/kvq4nvn/MSDS/capstone_project/results/counts   TRUE
      TPM Results      /home/kvq4nvn/MSDS/capstone_project/results/tpm   TRUE
   DESeq2 Results   /home/kvq4nvn/MSDS/capstone_project/results/deseq2   TRUE
  Pathway Results  /home/kvq4nvn/MSDS/capstone_project/results/pathway   TRUE
          Figures          /home/kvq4nvn/MSDS/capstone_project/figures   TRUE

✓ All expected directories exist.



In [24]:
###############################################################################
# SPONSOR DATA STRUCTURE CHECK
###############################################################################

sponsor_subdirs <- c(
  "metadata",
  "quants",
  "raw_reads",
  "reference"
)

sponsor_status <- data.frame(
  Directory = sponsor_subdirs,
  Path = file.path(base_dir, sponsor_subdirs),
  Exists = dir.exists(file.path(base_dir, sponsor_subdirs)),
  stringsAsFactors = FALSE
)

cat("SPONSOR DATA STATUS\n")
print(sponsor_status, row.names = FALSE)

sponsor_data_ok <- all(sponsor_status$Exists)

if (sponsor_data_ok) {
  cat("\n✓ Sponsor data structure is accessible.\n\n")
} else {
  cat("\n⚠ One or more sponsor data directories are unavailable.\n\n")
}

SPONSOR DATA STATUS
 Directory                                        Path Exists
  metadata  /standard/harrislab/capstone_2026/metadata   TRUE
    quants    /standard/harrislab/capstone_2026/quants   TRUE
 raw_reads /standard/harrislab/capstone_2026/raw_reads   TRUE
 reference /standard/harrislab/capstone_2026/reference   TRUE

✓ Sponsor data structure is accessible.



In [25]:
cat("\nChecking helper scripts...\n")


Checking helper scripts...


In [26]:
###############################################################################
# HELPER SCRIPT CHECK
###############################################################################

helper_files <- c(
  "metadata_helpers.R",
  "tximport_helpers.R",
  "deseq2_helpers.R",
  "plotting_helpers.R",
  "pathway_helpers.R"
)

helper_paths <- file.path(scripts_dir, helper_files)

helper_status <- data.frame(
  Helper = helper_files,
  Path = helper_paths,
  Exists = file.exists(helper_paths),
  Size_Bytes = ifelse(
    file.exists(helper_paths),
    file.info(helper_paths)$size,
    NA_real_
  ),
  stringsAsFactors = FALSE
)

cat("HELPER SCRIPT STATUS\n")
print(helper_status, row.names = FALSE)

helpers_exist_ok <- all(helper_status$Exists)

if (helpers_exist_ok) {
  cat("\n✓ All expected helper files exist.\n\n")
} else {
  cat("\n⚠ Missing helper files:\n")
  print(helper_status[!helper_status$Exists, ], row.names = FALSE)
  cat("\n")
}

HELPER SCRIPT STATUS
             Helper
 metadata_helpers.R
 tximport_helpers.R
   deseq2_helpers.R
 plotting_helpers.R
  pathway_helpers.R
                                                           Path Exists
 /home/kvq4nvn/MSDS/capstone_project/scripts/metadata_helpers.R   TRUE
 /home/kvq4nvn/MSDS/capstone_project/scripts/tximport_helpers.R   TRUE
   /home/kvq4nvn/MSDS/capstone_project/scripts/deseq2_helpers.R   TRUE
 /home/kvq4nvn/MSDS/capstone_project/scripts/plotting_helpers.R   TRUE
  /home/kvq4nvn/MSDS/capstone_project/scripts/pathway_helpers.R   TRUE
 Size_Bytes
       6171
          0
          0
          0
          0

✓ All expected helper files exist.



In [27]:
cat("\nChecking metadata...\n")


Checking metadata...


In [28]:
###############################################################################
# LOAD AND VALIDATE METADATA HELPER SCRIPT
###############################################################################

metadata_helper_path <- file.path(
  scripts_dir,
  "metadata_helpers.R"
)

metadata_helper_loaded <- FALSE
metadata_functions_ok <- FALSE

if (file.exists(metadata_helper_path)) {
  metadata_helper_loaded <- tryCatch(
    {
      source(metadata_helper_path)
      cat("✓ metadata_helpers.R loaded successfully.\n")
      TRUE
    },
    error = function(e) {
      cat("✗ metadata_helpers.R could not be loaded.\n")
      cat("  Error:", conditionMessage(e), "\n")
      FALSE
    }
  )
} else {
  cat("✗ metadata_helpers.R was not found.\n")
}

expected_metadata_functions <- c(
  "standardize_cols",
  "read_geo_sample_section",
  "read_gse146680_metadata",
  "read_gse203655_metadata",
  "read_gse212277_metadata",
  "build_master_sample_table"
)

metadata_function_status <- data.frame(
  Function = expected_metadata_functions,
  Exists = vapply(
    expected_metadata_functions,
    exists,
    logical(1),
    mode = "function"
  ),
  stringsAsFactors = FALSE
)

cat("\nMETADATA FUNCTION STATUS\n")
print(metadata_function_status, row.names = FALSE)

metadata_functions_ok <- all(metadata_function_status$Exists)

if (metadata_functions_ok) {
  cat("\n✓ All expected metadata functions are available.\n\n")
} else {
  cat("\n⚠ One or more metadata functions are unavailable.\n\n")
}


###############################################################################
# METADATA SOURCE FILE CHECK
###############################################################################

metadata_source_files <- c(
  "GSE146680/harris_GEO_spreadsheet.xls",
  "GSE203655/metadata_wt_vs_stat1_ko.xlsx",
  "GSE212277/sra_accession.xlsx"
)

metadata_source_paths <- file.path(
  base_dir,
  "metadata",
  metadata_source_files
)

metadata_source_status <- data.frame(
  Dataset = c("GSE146680", "GSE203655", "GSE212277"),
  File = metadata_source_files,
  Exists = file.exists(metadata_source_paths),
  stringsAsFactors = FALSE
)

cat("METADATA SOURCE FILE STATUS\n")
print(metadata_source_status, row.names = FALSE)

metadata_sources_ok <- all(metadata_source_status$Exists)

if (metadata_sources_ok) {
  cat("\n✓ All expected source metadata files exist.\n\n")
} else {
  cat("\n⚠ One or more source metadata files are missing.\n\n")
}

✓ metadata_helpers.R loaded successfully.

METADATA FUNCTION STATUS
                  Function Exists
          standardize_cols   TRUE
   read_geo_sample_section   TRUE
   read_gse146680_metadata   TRUE
   read_gse203655_metadata   TRUE
   read_gse212277_metadata   TRUE
 build_master_sample_table   TRUE

✓ All expected metadata functions are available.

METADATA SOURCE FILE STATUS
   Dataset                                   File Exists
 GSE146680   GSE146680/harris_GEO_spreadsheet.xls   TRUE
 GSE203655 GSE203655/metadata_wt_vs_stat1_ko.xlsx   TRUE
 GSE212277           GSE212277/sra_accession.xlsx   TRUE

✓ All expected source metadata files exist.



In [29]:
###############################################################################
# 9. METADATA SOURCE FILE CHECK
###############################################################################

metadata_source_files <- c(
  "GSE146680/harris_GEO_spreadsheet.xls",
  "GSE203655/metadata_wt_vs_stat1_ko.xlsx",
  "GSE212277/sra_accession.xlsx"
)

metadata_source_paths <- file.path(
  base_dir,
  "metadata",
  metadata_source_files
)

metadata_source_status <- data.frame(
  Dataset = c("GSE146680", "GSE203655", "GSE212277"),
  File = metadata_source_files,
  Exists = file.exists(metadata_source_paths),
  stringsAsFactors = FALSE
)

cat("METADATA SOURCE FILE STATUS\n")
print(metadata_source_status, row.names = FALSE)

metadata_sources_ok <- all(metadata_source_status$Exists)

if (metadata_sources_ok) {
  cat("\n✓ All expected source metadata files exist.\n\n")
} else {
  cat("\n⚠ One or more source metadata files are missing.\n\n")
}

METADATA SOURCE FILE STATUS
   Dataset                                   File Exists
 GSE146680   GSE146680/harris_GEO_spreadsheet.xls   TRUE
 GSE203655 GSE203655/metadata_wt_vs_stat1_ko.xlsx   TRUE
 GSE212277           GSE212277/sra_accession.xlsx   TRUE

✓ All expected source metadata files exist.



In [30]:
###############################################################################
# MASTER SAMPLE TABLE CHECK
###############################################################################

master_sample_path <- file.path(
  metadata_results_dir,
  "master_sample_table.csv"
)

master_sample_exists <- file.exists(master_sample_path)
master_sample_table <- NULL
master_table_ok <- FALSE

cat("MASTER SAMPLE TABLE STATUS\n")
cat("Path:", master_sample_path, "\n")
cat("Exists:", master_sample_exists, "\n")

if (master_sample_exists) {
  master_sample_table <- tryCatch(
    {
      read.csv(
        master_sample_path,
        stringsAsFactors = FALSE,
        check.names = FALSE
      )
    },
    error = function(e) {
      cat("✗ The Master Sample Table could not be read.\n")
      cat("  Error:", conditionMessage(e), "\n")
      NULL
    }
  )
}

required_master_columns <- c(
  "dataset",
  "sample",
  "quant_path",
  "file_exists",
  "cell_type",
  "condition",
  "genotype",
  "genotype_short",
  "comparison_group"
)

if (!is.null(master_sample_table)) {
  missing_master_columns <- setdiff(
    required_master_columns,
    names(master_sample_table)
  )

  cat("\nRows:", nrow(master_sample_table), "\n")
  cat("Columns:", ncol(master_sample_table), "\n")
  cat(
    "Datasets:",
    paste(unique(master_sample_table$dataset), collapse = ", "),
    "\n"
  )

  if (length(missing_master_columns) == 0) {
    cat("✓ All required Master Sample Table columns are present.\n")
  } else {
    cat(
      "✗ Missing required column(s):",
      paste(missing_master_columns, collapse = ", "),
      "\n"
    )
  }

  duplicate_sample_keys <- duplicated(
    paste(
      master_sample_table$dataset,
      master_sample_table$sample,
      sep = "::"
    )
  )

  cat(
    "Duplicate dataset/sample combinations:",
    sum(duplicate_sample_keys),
    "\n"
  )

  master_table_ok <- (
    nrow(master_sample_table) > 0 &&
    length(missing_master_columns) == 0 &&
    sum(duplicate_sample_keys) == 0
  )
}

cat("\n")

MASTER SAMPLE TABLE STATUS
Path: /home/kvq4nvn/MSDS/capstone_project/results/metadata/master_sample_table.csv 
Exists: TRUE 

Rows: 32 
Columns: 24 
Datasets: GSE146680, GSE203655, GSE212277 
✓ All required Master Sample Table columns are present.
Duplicate dataset/sample combinations: 0 



In [31]:
###############################################################################
# MASTER SAMPLE TABLE COMPLETENESS CHECK
###############################################################################

metadata_completeness_ok <- FALSE

if (!is.null(master_sample_table)) {
  important_fields <- c(
    "dataset",
    "sample",
    "quant_path",
    "cell_type",
    "condition",
    "genotype",
    "genotype_short",
    "comparison_group"
  )

  metadata_completeness <- data.frame(
    Field = important_fields,
    Missing = vapply(
      important_fields,
      function(field) {
        sum(
          is.na(master_sample_table[[field]]) |
          trimws(as.character(master_sample_table[[field]])) == ""
        )
      },
      numeric(1)
    ),
    stringsAsFactors = FALSE
  )

  cat("MASTER SAMPLE TABLE COMPLETENESS\n")
  print(metadata_completeness, row.names = FALSE)

  metadata_completeness_ok <- all(
    metadata_completeness$Missing == 0
  )

  if (metadata_completeness_ok) {
    cat("\n✓ Required metadata fields are complete.\n\n")
  } else {
    cat("\n⚠ Some required metadata fields contain missing values.\n\n")
  }
}


###############################################################################
# SAMPLE DISTRIBUTION CHECK
###############################################################################

sample_counts_ok <- FALSE

if (!is.null(master_sample_table)) {
  dataset_counts <- as.data.frame(
    table(master_sample_table$dataset),
    stringsAsFactors = FALSE
  )

  names(dataset_counts) <- c("Dataset", "Sample_Count")

  cat("SAMPLE COUNTS BY DATASET\n")
  print(dataset_counts, row.names = FALSE)

  expected_counts <- c(
    "GSE146680" = 8,
    "GSE203655" = 8,
    "GSE212277" = 16
  )

  observed_counts <- setNames(
    dataset_counts$Sample_Count,
    dataset_counts$Dataset
  )

  all_expected_present <- all(
    names(expected_counts) %in% names(observed_counts)
  )

  sample_counts_ok <- (
    all_expected_present &&
    all(
      observed_counts[names(expected_counts)] ==
      expected_counts
    )
  )

  if (sample_counts_ok) {
    cat("\n✓ Dataset sample counts match expectations.\n\n")
  } else {
    cat("\n⚠ Dataset sample counts differ from expectations.\n\n")
  }
}


###############################################################################
# SALMON QUANTIFICATION FILE CHECK
###############################################################################

quant_files_ok <- FALSE

if (!is.null(master_sample_table)) {
  quant_file_exists_now <- file.exists(
    master_sample_table$quant_path
  )

  quant_status <- data.frame(
    Dataset = master_sample_table$dataset,
    Sample = master_sample_table$sample,
    Quant_File_Exists = quant_file_exists_now,
    stringsAsFactors = FALSE
  )

  cat("SALMON QUANTIFICATION FILE STATUS\n")
  cat("Expected quant.sf files:", nrow(quant_status), "\n")
  cat("Files found:", sum(quant_status$Quant_File_Exists), "\n")
  cat("Files missing:", sum(!quant_status$Quant_File_Exists), "\n")

  if (any(!quant_status$Quant_File_Exists)) {
    cat("\nMissing quant.sf files:\n")
    print(
      quant_status[!quant_status$Quant_File_Exists, ],
      row.names = FALSE
    )
  }

  quant_files_ok <- all(quant_status$Quant_File_Exists)

  if (quant_files_ok) {
    cat("\n✓ All quant.sf files are available.\n\n")
  } else {
    cat("\n⚠ One or more quant.sf files are missing.\n\n")
  }
}


###############################################################################
# TRANSCRIPT-TO-GENE REFERENCE CHECK
###############################################################################

tx2gene_path <- file.path(
  base_dir,
  "reference",
  "Mus_musculus.GRCm39.transcript2gene.csv"
)

tx2gene_exists <- file.exists(tx2gene_path)
tx2gene <- NULL
reference_ok <- FALSE

cat("TRANSCRIPT-TO-GENE REFERENCE STATUS\n")
cat("Path:", tx2gene_path, "\n")
cat("Exists:", tx2gene_exists, "\n")

if (tx2gene_exists) {
  tx2gene <- tryCatch(
    {
      read.csv(
        tx2gene_path,
        stringsAsFactors = FALSE,
        check.names = FALSE
      )
    },
    error = function(e) {
      cat("✗ Reference file could not be read.\n")
      cat("  Error:", conditionMessage(e), "\n")
      NULL
    }
  )
}

required_reference_columns <- c(
  "transcript_id",
  "gene_id"
)

if (!is.null(tx2gene)) {
  missing_reference_columns <- setdiff(
    required_reference_columns,
    names(tx2gene)
  )

  cat("Rows:", nrow(tx2gene), "\n")
  cat("Columns:", ncol(tx2gene), "\n")

  if (length(missing_reference_columns) == 0) {
    cat("✓ Required reference columns are present.\n")
    reference_ok <- nrow(tx2gene) > 0
  } else {
    cat(
      "✗ Missing reference column(s):",
      paste(missing_reference_columns, collapse = ", "),
      "\n"
    )
  }
}

cat("\n")


###############################################################################
# OPTIONAL EXISTING OUTPUT CHECK
###############################################################################

expected_output_files <- c(
  "Master Sample Table" = master_sample_path,
  "Gene Count Matrix" = file.path(
    counts_dir,
    "gene_counts_all_samples.csv"
  ),
  "Gene TPM Matrix" = file.path(
    tpm_dir,
    "gene_tpm_all_samples.csv"
  ),
  "tximport RDS" = file.path(
    results_dir,
    "txi_all_samples.rds"
  )
)

output_status <- data.frame(
  Output = names(expected_output_files),
  Path = unname(expected_output_files),
  Exists = file.exists(expected_output_files),
  stringsAsFactors = FALSE
)

cat("EXISTING OUTPUT STATUS\n")
print(output_status, row.names = FALSE)

cat(
  "\nNote: Missing tximport outputs are acceptable if Notebook 02",
  "has not been run yet.\n\n"
)

MASTER SAMPLE TABLE COMPLETENESS
            Field Missing
          dataset       0
           sample       0
       quant_path       0
        cell_type       0
        condition       0
         genotype       0
   genotype_short       0
 comparison_group       0

✓ Required metadata fields are complete.

SAMPLE COUNTS BY DATASET
   Dataset Sample_Count
 GSE146680            8
 GSE203655            8
 GSE212277           16

✓ Dataset sample counts match expectations.

SALMON QUANTIFICATION FILE STATUS
Expected quant.sf files: 32 
Files found: 32 
Files missing: 0 

✓ All quant.sf files are available.

TRANSCRIPT-TO-GENE REFERENCE STATUS
Path: /standard/harrislab/capstone_2026/reference/Mus_musculus.GRCm39.transcript2gene.csv 
Exists: TRUE 
Rows: 116358 
Columns: 8 
✓ Required reference columns are present.

EXISTING OUTPUT STATUS
              Output
 Master Sample Table
   Gene Count Matrix
     Gene TPM Matrix
        tximport RDS
                                                 

In [32]:
###############################################################################
# 16. FINAL PROJECT HEALTH SUMMARY
###############################################################################

health_summary <- data.frame(
  Check = c(
    "Required packages",
    "Project directories",
    "Sponsor data directories",
    "Helper script files",
    "Metadata helper functions",
    "Source metadata files",
    "Master Sample Table structure",
    "Required metadata completeness",
    "Expected sample counts",
    "Salmon quant.sf files",
    "Transcript-to-gene reference"
  ),
  Passed = c(
    packages_ok,
    directories_ok,
    sponsor_data_ok,
    helpers_exist_ok,
    metadata_functions_ok,
    metadata_sources_ok,
    master_table_ok,
    metadata_completeness_ok,
    sample_counts_ok,
    quant_files_ok,
    reference_ok
  ),
  stringsAsFactors = FALSE
)

health_summary$Status <- ifelse(
  health_summary$Passed,
  "PASS",
  "FAIL"
)

cat("============================================================\n")
cat(" PROJECT HEALTH SUMMARY\n")
cat("============================================================\n")

print(
  health_summary[, c("Check", "Status")],
  row.names = FALSE
)

all_checks_passed <- all(health_summary$Passed)

cat("\n")

if (all_checks_passed) {
  cat("✓ PROJECT STATUS: READY FOR ANALYSIS\n")
  cat("You may continue to 02_tximport.ipynb.\n")
} else {
  failed_checks <- health_summary$Check[
    !health_summary$Passed
  ]

  cat("⚠ PROJECT STATUS: ATTENTION REQUIRED\n")
  cat(
    "Review the following failed check(s):\n-",
    paste(failed_checks, collapse = "\n- "),
    "\n"
  )
}

cat("============================================================\n")

 PROJECT HEALTH SUMMARY
                          Check Status
              Required packages   PASS
            Project directories   PASS
       Sponsor data directories   PASS
            Helper script files   PASS
      Metadata helper functions   PASS
          Source metadata files   PASS
  Master Sample Table structure   PASS
 Required metadata completeness   PASS
         Expected sample counts   PASS
          Salmon quant.sf files   PASS
   Transcript-to-gene reference   PASS

✓ PROJECT STATUS: READY FOR ANALYSIS
You may continue to 02_tximport.ipynb.
